In [1]:

from pathlib import Path
from collections import defaultdict
import hashlib
import shutil

source_path = Path("../data/skin_dataset")
clean_path = Path("../data/skin_dataset_clean")

splits = ["train", "valid", "test"]
classes = ["acne", "dry", "oil"]
extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# Stop rather than overwrite a previous cleaned dataset
if clean_path.exists():
    raise FileExistsError(
        f"{clean_path} already exists. Choose a new output folder "
        "or inspect the existing folder before proceeding."
    )

# Group identical file contents
hash_groups = defaultdict(list)

for split in splits:
    for class_name in classes:
        folder = source_path / split / class_name

        if not folder.exists():
            raise FileNotFoundError(f"Missing folder: {folder}")

        for path in folder.iterdir():
            if path.is_file() and path.suffix.lower() in extensions:
                digest = hashlib.sha256(path.read_bytes()).hexdigest()
                hash_groups[digest].append((split, class_name, path))

# Assign each image group to one split only.
# Prefer test, then valid, then train, so held-out examples
# stay held out when duplicates occur across splits.
priority = {"test": 0, "valid": 1, "train": 2}
kept = 0
removed_copies = 0

for digest, files in hash_groups.items():
    # All exact copies share one destination split.
    # If a group crosses splits, prioritize test, then valid.
    chosen_split = min(
        (item[0] for item in files),
        key=lambda s: priority[s]
    )

    # Keep the class and file from the chosen split.
    chosen_items = [item for item in files if item[0] == chosen_split]

    # If the same bytes occur under different class labels,
    # stop and inspect instead of silently choosing a label.
    labels = {item[1] for item in files}
    if len(labels) > 1:
        raise ValueError(
            f"Same image hash has conflicting labels: {files}"
        )

    split, class_name, chosen_path = chosen_items[0]
    destination = clean_path / split / class_name
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(chosen_path, destination / chosen_path.name)
    kept += 1
    removed_copies += len(files) - 1

print("Clean dataset created at:", clean_path)
print("Unique image groups kept:", kept)
print("Extra exact copies not copied:", removed_copies)


Clean dataset created at: ..\data\skin_dataset_clean
Unique image groups kept: 689
Extra exact copies not copied: 86


In [2]:

for split in splits:
    print(f"\n{split.upper()}")

    total = 0
    for class_name in classes:
        folder = clean_path / split / class_name
        count = sum(
            1 for p in folder.iterdir()
            if p.is_file() and p.suffix.lower() in extensions
        )
        total += count
        print(f"{class_name}: {count}")

    print("Total:", total)



TRAIN
acne: 207
dry: 198
oil: 198
Total: 603

VALID
acne: 20
dry: 19
oil: 19
Total: 58

TEST
acne: 10
dry: 9
oil: 9
Total: 28


In [3]:

from pathlib import Path
from collections import defaultdict
import hashlib

clean_path = Path("../data/skin_dataset_clean")
splits = ["train", "valid", "test"]
classes = ["acne", "dry", "oil"]
extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

hash_groups = defaultdict(list)

for split in splits:
    for class_name in classes:
        folder = clean_path / split / class_name

        for path in folder.iterdir():
            if path.is_file() and path.suffix.lower() in extensions:
                digest = hashlib.sha256(path.read_bytes()).hexdigest()
                hash_groups[digest].append((split, class_name, path.name))

cross_split_duplicates = {
    digest: files
    for digest, files in hash_groups.items()
    if len({item[0] for item in files}) > 1
}

print("Exact duplicate groups across splits:",
      len(cross_split_duplicates))


Exact duplicate groups across splits: 0
